# 03. Validation and Golden Set Design

- Parent issue: `#18`
- Status: `active`
- Summary: Define how the team will hold out validation data and protect against regressions.


## Audience and Why It Matters

Anyone evaluating results or reviewing training claims.


## Decision / Hypothesis

Reserve a stratified validation split and a smaller golden regression set before any serious training work starts.


## Environment and Reproduction

- Python: 3.7.9 (tested); 3.11+ recommended for full src/ compatibility
- Run from repo root: `jupyter notebook` from `CS4650-Nvidia-Nemotron-Challenge/`
- Notebook uses `Path.cwd().parent` to resolve repo root when launched from `notebooks/`
- Input: `data/eval/train_normalized.jsonl` (produced by notebook 02)
- Outputs: `data/eval/validation_200.jsonl`, `data/eval/golden_20.jsonl`, `data/eval/splits_manifest.json` (all git-ignored)
- Companion issue and registry entry: `docs/execution/NOTEBOOKS.md`

In [1]:
import sys, json, random, platform
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SEED = 42
random.seed(SEED)

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")
print(f"Platform  : {platform.platform()}")
print(f"Seed      : {SEED}")

Repo root : C:\Users\SBaroni\CS4650-Nvidia-Nemotron-Challenge
Python    : 3.7.9
Platform  : Windows-10-10.0.26100-SP0
Seed      : 42


## Method and Outputs

            ### Planned method
            - Document the validation split policy and category coverage expectations.
- Describe the golden-set role in catching regressions that aggregate metrics can hide.
- Tie these choices back to the plan review findings.

            ### Expected outputs
            - Validation split policy
- Golden-set checklist
- Measurement caveats for small category slices


In [2]:
IN_PATH = REPO_ROOT / "data" / "eval" / "train_normalized.jsonl"

records = []
with open(IN_PATH) as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} records from train_normalized.jsonl")
print(f"Sample: {records[0]}")

Loaded 9500 records from train_normalized.jsonl
Sample: {'example_id': '00066667', 'category': 'bit_manipulation', 'prompt': "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100", 'gold': '10010111', 'source': 'kaggle-official', 'split': 'train', 'dataset_version': 'kaggle-v1', 'metadata': {}}


In [3]:
from collections import defaultdict

# Group records by category
by_category = defaultdict(list)
for rec in records:
    by_category[rec["category"]].append(rec)

categories = sorted(by_category.keys())
print(f"Categories: {categories}")

# Sample evenly: 200 rows / 6 categories = ~33 per category
N_VAL = 200
per_category = N_VAL // len(categories)  # 33
remainder = N_VAL % len(categories)       # 2 leftover

val_rows = []
for i, cat in enumerate(categories):
    pool = by_category[cat].copy()
    random.shuffle(pool)
    # give 1 extra row to the first `remainder` categories
    n = per_category + (1 if i < remainder else 0)
    val_rows.extend(pool[:n])

# Shuffle the final val set
random.shuffle(val_rows)

print(f"\nValidation split: {len(val_rows)} rows")
for cat in categories:
    count = sum(1 for r in val_rows if r["category"] == cat)
    print(f"  {cat:<25}: {count}")

Categories: ['bit_manipulation', 'equation_symbolic', 'numeral_system', 'physics_gravity', 'text_cipher', 'unit_conversion']

Validation split: 200 rows
  bit_manipulation         : 34
  equation_symbolic        : 34
  numeral_system           : 33
  physics_gravity          : 33
  text_cipher              : 33
  unit_conversion          : 33


In [4]:
VAL_PATH = REPO_ROOT / "data" / "eval" / "validation_200.jsonl"

with open(VAL_PATH, "w") as f:
    for rec in val_rows:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(val_rows)} rows → data/eval/validation_200.jsonl")

Wrote 200 rows → data/eval/validation_200.jsonl


In [5]:
# Get the example_ids already used in val so we don't overlap
val_ids = {r["example_id"] for r in val_rows}

# Sample from the remaining rows (not in val)
remaining = defaultdict(list)
for rec in records:
    if rec["example_id"] not in val_ids:
        remaining[rec["category"]].append(rec)

N_GOLDEN = 20
per_cat_golden = N_GOLDEN // len(categories)   # 3
remainder_golden = N_GOLDEN % len(categories)  # 2 leftover

golden_rows = []
for i, cat in enumerate(categories):
    pool = remaining[cat].copy()
    random.shuffle(pool)
    n = per_cat_golden + (1 if i < remainder_golden else 0)
    golden_rows.extend(pool[:n])

print(f"Golden set: {len(golden_rows)} rows")
for cat in categories:
    count = sum(1 for r in golden_rows if r["category"] == cat)
    print(f"  {cat:<25}: {count}")

# Confirm no overlap with val
overlap = {r["example_id"] for r in golden_rows} & val_ids
print(f"\nOverlap with val set: {len(overlap)} (should be 0)")

Golden set: 20 rows
  bit_manipulation         : 4
  equation_symbolic        : 4
  numeral_system           : 3
  physics_gravity          : 3
  text_cipher              : 3
  unit_conversion          : 3

Overlap with val set: 0 (should be 0)


In [6]:
GOLDEN_PATH = REPO_ROOT / "data" / "eval" / "golden_20.jsonl"

with open(GOLDEN_PATH, "w") as f:
    for rec in golden_rows:
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(golden_rows)} rows → data/eval/golden_20.jsonl")
print("WARNING: golden_20.jsonl is now frozen. Do not edit in place.")
print("If changes are needed, create golden_40.jsonl instead.")

Wrote 20 rows → data/eval/golden_20.jsonl
If changes are needed, create golden_40.jsonl instead.


In [7]:
import datetime

manifest = {
    "created_at": datetime.datetime.utcnow().isoformat() + "Z",
    "dataset_version": "kaggle-v1",
    "seed": SEED,
    "source": "data/eval/train_normalized.jsonl",
    "validation": {
        "filename": "validation_200.jsonl",
        "n_rows": len(val_rows),
        "per_category": {cat: sum(1 for r in val_rows if r["category"] == cat) for cat in categories}
    },
    "golden": {
        "filename": "golden_20.jsonl",
        "n_rows": len(golden_rows),
        "per_category": {cat: sum(1 for r in golden_rows if r["category"] == cat) for cat in categories}
    }
}

MANIFEST_PATH = REPO_ROOT / "data" / "eval" / "splits_manifest.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest:")
print(json.dumps(manifest, indent=2))

Manifest:
{
  "created_at": "2026-05-01T21:49:37.347338Z",
  "dataset_version": "kaggle-v1",
  "seed": 42,
  "source": "data/eval/train_normalized.jsonl",
  "validation": {
    "filename": "validation_200.jsonl",
    "n_rows": 200,
    "per_category": {
      "bit_manipulation": 34,
      "equation_symbolic": 34,
      "numeral_system": 33,
      "physics_gravity": 33,
      "text_cipher": 33,
      "unit_conversion": 33
    }
  },
  "golden": {
    "filename": "golden_20.jsonl",
    "n_rows": 20,
    "per_category": {
      "bit_manipulation": 4,
      "equation_symbolic": 4,
      "numeral_system": 3,
      "physics_gravity": 3,
      "text_cipher": 3,
      "unit_conversion": 3
    }
  }
}


## Results / Open Risks

- Produced `validation_200.jsonl`: 200 rows, stratified across 6 categories (~33 per category), seed=42.
- Produced `golden_20.jsonl`: 20 rows, ~3-4 per category, no overlap with val set, seed=42.
- Manifest written to `data/eval/splits_manifest.json` recording provenance.
- Risk: 33 rows per category is a small sample — accuracy scores on individual categories will have high variance. Use overall accuracy as the primary metric.
- Risk: golden_20.jsonl is now frozen. Any future changes require a new versioned filename.

## Sources

- [Execution plan v0.2](../docs/planning/plan_v0.2.md)
- [Architecture doc](../docs/architecture/ARCHITECTURE.md)
- [data/eval/README.md](../data/eval/README.md)
- [notebooks/02_dataset_schema_and_eda.ipynb](02_dataset_schema_and_eda.ipynb)